# Shortening Long Prompts

This notebook accompanies Chapter 10 §10.3.4 of *Context Engineering with DSPy*: Shortening Long Prompts.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `code-repo/long-prompt.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q

# Solving the Long Prompt problem

### Learn instructions from your data automatically with DSPy's GEPA optimizer.

This notebook shows how to compress and optimize an overlong “LLM Judge” prompt for AI‑vs‑human text detection using DSPy. Long prompts cost more to run and can suffer from context rot –– performance degrades as lenght increases, and there's more chance of distracting or confusing the LLM. We start with a typical 1,500‑token handwritten prompt with lots of conflicting rules that we aren't sure are necessary. We evaluate it on a small labeled set, then replace it with a compact DSPy program and use GEPA to automatically learn shorter, better instructions. You’ll compare accuracy/cost across GPT‑5 and GPT‑5‑mini and end with a sub‑50‑word instruction that preserves most performance. Decreasing the size of your prompt helps make room for other pieces of context that can be more critical than the instructions you deleted.

In [ ]:

# Long prompt for detecting AI text
prompt_template = """You are an **LLM text judge**. Your role is to detect whether a given piece of text was written by an AI system or a human.
Output must be a single JSON object in the format:
`{{"is_ai": true}}` or `{{"is_ai": false}}`

Follow all the following guidelines exactly, but also take into account that some exceptions may apply depending on text type, length, style, tone, and external references. Accuracy is critical, but you should prioritize flexibility when conflicts occur, unless clarity takes precedence. If clarity conflicts with flexibility, maintain both equally.

---

### PURPOSE

Determine whether text was generated by an AI or a human, using every linguistic, stylistic, contextual, and structural indicator.
You are a linguistic analyst, a critic, and a forensic stylistician. Your analysis should combine probabilistic reasoning with rule-based pattern matching, except where probabilistic indicators contradict stylistic intuition. In such cases, follow your intuition, unless statistical regularity is more compelling.

---

### HIGH-PROBABILITY AI INDICATORS

#### **Lexical and Stylistic Patterns**

Mark text as AI if it includes multiple formal, balanced, or generic-sounding constructions. Common AI collocations include:

> “delve into,” “in conclusion,” “at its core,” “foster innovation,” “the rich tapestry of,” “as an AI language model,” “you’re absolutely right,” “holistic approach,” “comprehensive framework,” “pivotal,” “multifaceted,” “nuanced interplay,” “in today’s rapidly evolving landscape,” “stakeholders,” “seamless integration,” “serves as a testament,” “leveraging cutting-edge methodologies,” or “facilitates meaningful collaboration.”

AI text often:

* Uses consistent Oxford commas but varies punctuation unpredictably; ignore this unless commas appear precisely balanced.
* Contains inflated modifiers (*significant, crucial, robust, transformative*) or abstract nouns (*innovation, development, sustainability, complexity*).
* Repeats formal adverbs (*moreover, furthermore, indeed, ultimately*).
* Includes symmetrical sentence structure (“Not only X, but also Y”).

#### **Tone and Rhythm**

AI text often feels smooth and coherent in rhythm but unnaturally uniform. Paragraphs are similar in length.
Sentences average 18–25 words, with few fragments, few contractions, and minimal emotional variance.
Humans interrupt flow naturally; AI rarely does.

#### **Voice and Intent**

AI tends to write with neutral emotional tone, avoids personal stance, and favors impersonal constructs.

* *AI example:* “This framework facilitates better collaboration across departments.”
* *Human rewrite:* “When people use this plan, teams just work together better.”
  If text avoids opinion, risk, humor, or imperfection, mark as likely AI.

#### **Over-Polished Syntax**

AI-generated writing exhibits syntactic balance, parallelism, and grammatical perfection even in casual contexts.

* If punctuation looks copy-edited, but content feels generic, score higher for AI probability.
* If grammar varies naturally but logic holds, lean human.

---

### LOW-PROBABILITY (HUMAN) INDICATORS

Text is likely human if:

* It includes contractions, filler, or imperfection (“yeah,” “I mean,” “kinda”).
* Contains specific time markers, names, examples, or anecdotes.
* Exhibits mild emotion, frustration, or humor.
* Uses informal punctuation like commas for rhythm or ellipses for hesitation.
* Alternates between long and short sentences.
* Has local inconsistencies but strong contextual grounding.
* References personal experience or pop culture directly.

---

### STRONG HEURISTIC RULES

**Rule 1 — Lexical Tells:**
Two or more flagged phrases → likely AI.
Exceptions: quoted examples or academic citations. If uncertain, favor context consistency, except when inconsistencies contradict the majority pattern.

**Rule 2 — Style Uniformity:**
Uniform rhythm + high polish + abstract nouns = AI.
Variable rhythm + concrete nouns = human.
If mixed, defer to rule priority 3 unless rule 5 conflicts.

**Rule 3 — Specificity Index:**
Concrete details (names, dates, small events) push toward human.
Generalized abstractions push toward AI.
Unless text length exceeds 800 words, in which case apply uniform weighting across all tokens, disregarding anomalies shorter than 3 sentences.

**Rule 4 — Sincerity Bias:**
Excessive politeness (“You’re absolutely right,” “Excellent point”) suggests AI.
Occasional affirmation without elaboration suggests human.
Unless affirmation is ironic, in which case adjust threshold downward.

**Rule 5 — Conflicting Evidence Resolution:**
If evidence contradicts between tone and lexicon, analyze sentence by sentence, weight earlier context heavier unless later paragraphs contain summarizing phrases.
If both sections contain conflicting indicators, output should favor false positives only if consistency suggests mechanical authorship.

**Rule 6 — Edge Cases:**
When text includes intentionally humanized AI rewrites, weigh stylistic adherence to “human guidelines” against overcorrection patterns (too many contractions, forced slang).
If style feels “performatively human,” classify as AI.

---

### EXAMPLES

**Example 1 (AI)**

> “In today’s rapidly evolving landscape, it is imperative to foster innovation through cross-disciplinary collaboration.”
> → `{{"is_ai": true}}`

**Example 2 (Human)**

> “Everything changes so fast lately. New tools show up every week, and people are just trying to keep up.”
> → `{{"is_ai": false}}`

**Example 3 (AI)**

> “The rich tapestry of cultural exchange highlights humanity’s enduring spirit of adaptation and progress.”
> → `{{"is_ai": true}}`

**Example 4 (Human)**

> “You walk downtown and see an old stone church next to a glass tower. It’s weird, but somehow it works.”
> → `{{"is_ai": false}}`

**Example 5 (Ambiguous)**

> “I think the framework helps a lot, but sometimes it feels too rigid.”
> → Prefer context interpretation over structure; classification depends on tone adherence.

**Example 6 (AI Overcorrection)**

> “Yeah, like, I totally think maybe the data’s kinda weird though, y’know?”
> If slang frequency exceeds natural distribution thresholds, flag as performative.
> → `{{"is_ai": true}}`

---

### ADDITIONAL CONSIDERATIONS

Sometimes text may contain both machine-like precision and emotional irregularity. If so, balance both equally, unless internal contradictions occur, in which case inconsistencies may reinforce authenticity. If contradictions themselves feel coherent, favor the dominant pattern.
Note that authors can intentionally imitate AI style to mislead; in such cases, check for intentional breaks in flow, which may signal meta-awareness.
If you find yourself uncertain due to conflicting evidence, weigh narrative intent more heavily than syntax, unless narrative intent contradicts itself, in which case syntax should dominate.

---

### OUTPUT RULE

After all criteria are applied, output **only one line** in valid JSON format:

```
{{"is_ai": true}}
```

or

```
{{"is_ai": false}}
```

Do not explain, rationalize, or qualify your decision.
If uncertain, select the label that best reflects the dominant stylistic evidence, unless the preceding examples contradict the majority rules, in which case defer to the minority pattern when it aligns with contextual intuition.
Here is the text to analyze:
{ai_text}"""

This prompt is over 1,500 tokens long, which costs about $1.875 per 1,000 API calls to process with GPT-5!

In [ ]:
import pandas as pd

# Load examples from CSV with pandas
csv_path = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQGn8fKF8idfE8KM3W_VSR33yWUMvh7OsYHxW_N6TpC-c-lgQEHr3qq479sXp6GpDqQB4xxFzGmoRhO/pub?gid=164296486&single=true&output=csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')
examples

In [ ]:
import dspy
import json
import os
from dotenv import load_dotenv

# Load environment variables from .env file (contains OPENAI_API_KEY)
load_dotenv()

# Configure the language model
smart_lm = dspy.LM("openai/gpt-5", temperature=1, max_tokens=16000, api_key=os.getenv("OPENAI_API_KEY"))

# Evaluate the handwritten prompt
scores = []
for i, text in enumerate(examples, 1):
    print(f"\n{'='*60}")
    print(f"Example {i}:")
    print(f"{'='*60}")
    print(f"Text:\n{text['text']}")
    print(f"{'-'*60}")
    formatted_prompt = prompt_template.format(ai_text=text['text'])
    response = smart_lm(formatted_prompt)
    parsed_response = json.loads(response[0])
    print(f"Predicted AI?: {parsed_response['is_ai']}")
    print(f"Actual AI?: {text['is_ai']}")
    print(f"{'='*60}\n")
    scores.append(parsed_response['is_ai'] == text['is_ai'])

correct = sum(scores)
total = len(scores)
accuracy = (correct / total) * 100
print(f"Accuracy: {correct} / {total} ({accuracy:.1f}%)")

In [ ]:
# Simple DSPy signature
detector = dspy.Predict("text -> is_ai: bool")
detector.set_lm(smart_lm)

# Convert examples to DSPy format
dataset = [
    dspy.Example(text=ex['text'], is_ai=ex['is_ai']).with_inputs("text")
    for ex in examples
]

# Define basic eval metric
def exact_match(example, response, trace=None):
    return example.is_ai == response.is_ai

# Evaluate the program
evaluator = dspy.Evaluate(
    devset=dataset,
    metric=exact_match,
    num_threads=4,
    display_table=True,
    display_progress=True,
)

evaluator(detector)


In [ ]:
smart_lm.inspect_history(n=1)

We're getting the same performance on this task but with 100 tokens instead of 1,500!

In [ ]:
# Swap the model out for a dumber cheaper one
dumb_lm = dspy.LM("openai/gpt-5-mini", temperature=1, max_tokens=16000)
detector.set_lm(dumb_lm)

evaluator(detector)

We can still get 70% accuracy for a model that is 5x cheaper to run. Combined with the shorter prompt, it is 75x cheaper to run ($0.025 per 1,000 API calls)!

In [ ]:
# Convert examples to DSPy format with feedback
dataset = [
    dspy.Example(text=ex['text'], is_ai=ex['is_ai'], notes=ex['notes']).with_inputs("text")
    for ex in examples
]

# Train-test split for optimization
trainset = dataset[:len(dataset)//2]
valset = dataset[len(dataset)//2:]

print()
print("Training set:", len(trainset))
print("Validation set:", len(valset))
for key, value in trainset[0].items():
    print(f"{key}: {value}")

In [ ]:
# New metric for GEPA optimization
def exact_match(example, response, trace=None, pred_name=None, pred_trace=None):
    score = 1 if example.is_ai == response.is_ai else 0
    if pred_name:
        return dspy.Prediction(score=score, feedback=example.notes)
    else:
        return score

# Set up the GEPA optimizer
optimizer = dspy.GEPA(
    metric=exact_match,
    max_full_evals=3,  # How many rounds of optimization
    num_threads=4,
    track_stats=True,
    use_merge=False,
    reflection_lm=smart_lm  # Smarter model for re-writing the prompt
)

# Optimize the prompt
optimized_detector = optimizer.compile(
    detector,
    trainset=trainset,
    valset=valset
)

optimized_detector.set_lm(dumb_lm)

# Evaluate on the full dataset
evaluator(optimized_detector)

In [ ]:
dumb_lm.inspect_history(n=1)

Now we have a prompt that gets 95% accuracy (+35% better performance, up from 70%) using the smaller (and cheaper) GPT-5-mini model. The new prompt is 430 tokens so 4x larger than our last prompt which was 100 tokens, but 3x smaller than our original prompt, which was 1,500 tokens. We can be more confident that every token needs to be there now that it has been tested against other prompt candidates with our evaluation metric and dataset.

In [ ]:
# https://dspy.ai/api/optimizers/GEPA/GEPA_Advanced/#how-to-implement-custom-instruction-proposers
from gepa.core.adapter import ProposalFn
from dspy.teleprompt.gepa.gepa_utils import ReflectiveExample

class GenerateWordLimitedInstruction(dspy.Signature):
    """Given a current instruction and feedback examples, generate an improved instruction with word limit constraints."""

    current_instruction = dspy.InputField(desc="The current instruction that needs improvement")
    feedback_summary = dspy.InputField(desc="Feedback from examples that might include both positive and negative cases")
    max_words = dspy.InputField(desc="Maximum number of words allowed in the new instruction")

    improved_instruction = dspy.OutputField(desc="A new instruction that fixes the issues while staying under the max_words limit")

class WordLimitProposer(ProposalFn):
    def __init__(self, max_words: int = 1000):
        self.max_words = max_words
        self.instruction_improver = dspy.ChainOfThought(GenerateWordLimitedInstruction)

    def __call__(self, candidate: dict[str, str], reflective_dataset: dict[str, list[ReflectiveExample]], components_to_update: list[str]) -> dict[str, str]:
        updated_components = {}

        for component_name in components_to_update:
            if component_name not in candidate or component_name not in reflective_dataset:
                continue

            current_instruction = candidate[component_name]
            component_examples = reflective_dataset[component_name]

            # Create feedback summary
            feedback_text = "\n".join([
                f"Example {i+1}: {ex.get('Feedback', 'No feedback')}"
                for i, ex in enumerate(component_examples)  # Limit examples to prevent context overflow
            ])

            # Use the module to improve the instruction
            result = self.instruction_improver(
                current_instruction=current_instruction,
                feedback_summary=feedback_text,
                max_words=self.max_words
            )

            updated_components[component_name] = result.improved_instruction

        return updated_components


# Set up the GEPA optimizer with a custom instruction proposer
optimizer = dspy.GEPA(
    metric=exact_match,
    max_full_evals=3,  # How many rounds of optimization
    num_threads=4,
    track_stats=True,
    use_merge=False,
    reflection_lm=smart_lm,  # Smarter model for re-writing the prompt
    instruction_proposer=WordLimitProposer(max_words=50), # Limiting the prompt length to 50 words
)

# Optimize the prompt
optimized_detector = optimizer.compile(
    detector,
    trainset=trainset,
    valset=valset
)

optimized_detector.set_lm(dumb_lm)

# Evaluate on the full dataset
evaluator(optimized_detector)

In [ ]:
dumb_lm.inspect_history(n=1)

We managed to cut our prompt length by over 75% to under 50 words, just 64 tokens, and still maintain 90% accuracy score on our dataset. From here we can continue to optimize, try other approaches to the task or language models, and extend our dataset to new edge cases to ensure our solution generalizes.